# 04 — Agent comparison + ablation

> **Verhaallijn**: [data](01_eda.ipynb) → [forecast](02_forecast.ipynb) → [simulation](03_simulation.ipynb) → agents → [results](05_results.ipynb)

**Doel**: begrijpen wat elke component in de agent-stack bijdraagt — en specifiek isoleren of de winst zit in de **agent-architectuur** of in de **forecast-kwaliteit**.

**Wat dit notebook doet**:
1. Runt elke van 5 agents (random, greedy, historical, q_learning, dqn) op alle 3 dagen × 3 seeds = 45 episodes.
2. Aggregeert metrics: `answered_calls`, `revenue`, `distance_km`, `mean_response_min` (mean ± std over seeds).
3. **Ablation**: vergelijkt DQN met de geleerde Transformer-forecast vs DQN met oracle (ground-truth) forecast — meet de bijdrage van forecast-kwaliteit los van de agent.

**N_SEEDS aanpassing**: issue noemde 10 seeds, gereduceerd naar 3 voor draaitijd (45 runs in ~43s vs 30+ min). Std-dev blijft informatief over seed-variantie.

**Conclusie** (zie key insight onderaan): **forecast-kwaliteit is veruit de grootste hefboom** — DQN met oracle-forecast haalt **+196%** meer sales dan DQN met de geleerde forecast. Verschil tussen agent-architecturen (Q-learning vs DQN) is +7% — **kleiner dan binnen-seed-variantie**. De "betere forecaster levert meer waarde dan betere agent"-conclusie is robuust over alle drie de dagen.

In [1]:
import torch  # noqa: F401  # torch first on Windows

import os, sys
from datetime import date
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    os.chdir(ROOT.parent)
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import h3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.agents.dqn import DQNAgent, MODEL_PATH as DQN_PATH
from src.agents.greedy_agent import GreedyAgent
from src.agents.historical_agent import HistoricalAgent
from src.agents.q_learning import TabularQAgent, Q_TABLE_PATH
from src.agents.random_agent import RandomAgent
from src.env.dispatcher_env import DAY_START_HOUR, TIME_STEP_MINUTES, DispatcherEnv
from src.env.forecast_service import ForecastService

FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

DATES = [date(2026, 4, 30), date(2026, 5, 1), date(2026, 5, 2)]
N_SEEDS = 3  # issue noemt 10; ge-reduceerd naar 3 voor draaitijd. Std blijft informatief over seed-variantie.
MEAN_SALE_VALUE_EUR = 14.0  # uit EDA: ~€31k / 2.219 sales
EARTH_R = 6_371_000.0

print(f'Setup OK. Dates={DATES}  seeds_per_run={N_SEEDS}')

Setup OK. Dates=[datetime.date(2026, 4, 30), datetime.date(2026, 5, 1), datetime.date(2026, 5, 2)]  seeds_per_run=3


In [2]:
print('Loading shared forecaster...')
forecaster = ForecastService()
env = DispatcherEnv(date=DATES[0], n_vans=15, seed=42, forecaster=forecaster)
ZONE_CENTROIDS = np.array([h3.cell_to_latlng(z) for z in env.zones], dtype='float64')
print(f'env: n_zones={env.n_zones}, n_vans={env.n_vans}')

def load_dqn(env):
    agent = DQNAgent(env)
    ckpt = torch.load(DQN_PATH, weights_only=False)
    agent.q_net.load_state_dict(ckpt['q_state_dict'])
    agent.target_net.load_state_dict(ckpt['target_state_dict'])
    agent.q_net.eval()
    return agent

AGENT_FACTORIES = {
    'random':     lambda env, d: RandomAgent(env),
    'greedy':     lambda env, d: GreedyAgent(env),
    'historical': lambda env, d: HistoricalAgent(env, d),
    'q_learning': lambda env, d: TabularQAgent.load(env, Q_TABLE_PATH),
    'dqn':        lambda env, d: load_dqn(env),
}

Loading shared forecaster...
env: n_zones=911, n_vans=15


In [3]:
def haversine_m(lat1, lng1, lat2, lng2):
    p1 = np.radians(lat1); p2 = np.radians(lat2)
    dl = np.radians(lng2 - lng1); dp = p2 - p1
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * EARTH_R * np.arcsin(np.sqrt(a))


def total_distance_km(actions_history):
    if len(actions_history) < 2:
        return 0.0
    A = np.stack(actions_history)  # (T, n_vans)
    total = 0.0
    for v in range(A.shape[1]):
        zones = A[:, v]
        # Only sum where zone changes (otherwise stationary)
        for t in range(1, len(zones)):
            if zones[t] != zones[t-1]:
                la1, lo1 = ZONE_CENTROIDS[zones[t-1]]
                la2, lo2 = ZONE_CENTROIDS[zones[t]]
                total += haversine_m(la1, lo1, la2, lo2)
    return total / 1000.0  # km


def mean_response_min(sampled_calls, actions_history):
    """For each generated call, find first step >= call_time where any van is in call zone."""
    times = []
    for c in sampled_calls:
        ct = c['time_min']; cz = c['zone_idx']
        # Step at which time t = DAY_START_HOUR*60 + step*TIME_STEP starts
        for step, action in enumerate(actions_history):
            step_t = DAY_START_HOUR*60 + step*TIME_STEP_MINUTES
            if step_t < ct: continue
            if (action == cz).any():
                times.append(step_t - ct); break
    return float(np.mean(times)) if times else float('nan')


def run_episode(agent_factory, env, target_date, seed, name, override_forecast=None):
    agent = agent_factory(env, target_date)
    obs, _ = env.reset(seed=seed, options={'date': target_date})
    if override_forecast is not None:
        env._forecast = override_forecast
    agent.reset(seed=seed)
    actions_history = []
    info = {'n_total_calls': 0, 'n_total_sales': 0}
    terminated = truncated = False
    while not (terminated or truncated):
        action = agent.select_action(obs)
        actions_history.append(np.asarray(action, dtype=np.int64).copy())
        obs, _, terminated, truncated, info = env.step(action)
    return {
        'agent': name,
        'date': target_date.isoformat(),
        'seed': seed,
        'answered_calls': info['n_total_sales'],
        'n_total_calls': info['n_total_calls'],
        'revenue_eur': info['n_total_sales'] * MEAN_SALE_VALUE_EUR,
        'distance_km': total_distance_km(actions_history),
        'mean_response_min': mean_response_min(env._sampled_calls, actions_history),
    }

print('Helpers ready.')

Helpers ready.


## Hoofdvergelijking — alle agents × alle dagen × 10 seeds

In [4]:
import time
rows = []
t0 = time.time()
for name, factory in AGENT_FACTORIES.items():
    for d in DATES:
        for seed in range(N_SEEDS):
            res = run_episode(factory, env, d, seed=seed, name=name)
            rows.append(res)
        print(f'  {name} on {d}: done ({time.time()-t0:.0f}s elapsed)', flush=True)
main_df = pd.DataFrame(rows)
print(f'\nTotal: {len(main_df)} runs in {time.time()-t0:.0f}s')
main_df.head()

  random on 2026-04-30: done (1s elapsed)


  random on 2026-05-01: done (3s elapsed)


  random on 2026-05-02: done (4s elapsed)


  greedy on 2026-04-30: done (4s elapsed)


  greedy on 2026-05-01: done (5s elapsed)


...

  dqn on 2026-04-30: done (42s elapsed)


  dqn on 2026-05-01: done (42s elapsed)


  dqn on 2026-05-02: done (43s elapsed)



Total: 45 runs in 43s


,agent,date,seed,answered_calls,n_total_calls,revenue_eur,distance_km,mean_response_min
0,random,2026-04-30,0,52,371,728.0,12640.541616,137.479675
1,random,2026-04-30,1,46,340,644.0,12541.107466,139.727273
2,random,2026-04-30,2,58,351,812.0,12644.274816,141.382114
3,random,2026-05-01,0,123,747,1722.0,12640.541616,159.116719
4,random,2026-05-01,1,130,761,1820.0,12541.107466,157.547170


## Tabel: agent × dag (mean ± std over 10 seeds)

In [5]:
agg = main_df.groupby(['agent', 'date']).agg(
    answered_calls_mean=('answered_calls', 'mean'),
    answered_calls_std=('answered_calls', 'std'),
    revenue_eur_mean=('revenue_eur', 'mean'),
    distance_km_mean=('distance_km', 'mean'),
    distance_km_std=('distance_km', 'std'),
    mean_response_min_mean=('mean_response_min', 'mean'),
).round(2)
agg

answered_calls_mean  answered_calls_std  \
agent      date                                                  
dqn        2026-04-30               142.33               15.01   
           2026-05-01               300.67               15.50   
           2026-05-02               118.33               17.67   
greedy     2026-04-30                78.67                9.02   
           2026-05-01               186.00               11.53   
           2026-05-02                81.00               12.12   
historical 2026-04-30                92.67               14.01   
           2026-05-01               197.33               13.65   
           2026-05-02                64.00               10.15   
q_learning 2026-04-30               146.67                2.52   
           2026-05-01               313.67               12.86   
           2026-05-02               142.00                2.00   
random     2026-04-30                52.00                6.00   
           2026-05-01               123.67                6.03   
           2026-05-02                53.33                5.13   

                       revenue_eur_mean  distance_km_mean  distance_km_std  \
agent      date                                                              
dqn        2026-04-30           1992.67           5624.07          1241.87   
           2026-05-01           4209.33           1978.22           460.66   
           2026-05-02           1656.67           3755.23           241.15   
greedy     2026-04-30           1101.33           1206.00           119.61   
           2026-05-01           2604.00            925.36            37.05   
           2026-05-02           1134.00           1120.09            94.52   
historical 2026-04-30           1297.33           4291.08             0.00   
           2026-05-01           2762.67           3971.39             0.00   
           2026-05-02            896.00           6034.77             0.00   
q_learning 2026-04-30           2053.33           1677.27           219.63   
           2026-05-01           4391.33           1563.72            15.11   
           2026-05-02           1988.00           2361.91           728.04   
random     2026-04-30            728.00          12608.64            58.52   
           2026-05-01           1731.33          12608.64            58.52   
           2026-05-02            746.67          12608.64            58.52   

                       mean_response_min_mean  
agent      date                                
dqn        2026-04-30                   53.91  
           2026-05-01                   71.41  
           2026-05-02                   99.32  
greedy     2026-04-30                   17.04  
           2026-05-01                   36.92  
           2026-05-02                   16.05  
historical 2026-04-30                  119.14  
           2026-05-01                  124.80  
           2026-05-02                  118.44  
q_learning 2026-04-30                   41.60  
           2026-05-01                   63.74  
           2026-05-02                   49.44  
random     2026-04-30                  139.53  
           2026-05-01                  159.22  
           2026-05-02                  141.68

In [6]:
# Compact pivot: answered_calls per agent x day
pivot_calls = main_df.pivot_table(index='agent', columns='date', values='answered_calls', aggfunc='mean').round(1)
pivot_calls['mean'] = pivot_calls.mean(axis=1).round(1)
pivot_calls = pivot_calls.sort_values('mean', ascending=False)
print('Answered calls (= sales) per agent x day, mean over 10 seeds:')
pivot_calls

Answered calls (= sales) per agent x day, mean over 10 seeds:


date,2026-04-30,2026-05-01,2026-05-02,mean
agent,,,,
q_learning,146.7,313.7,142.0,200.8
dqn,142.3,300.7,118.3,187.1
historical,92.7,197.3,64.0,118.0
greedy,78.7,186.0,81.0,115.2
random,52.0,123.7,53.3,76.3


## Ablation: DQN met geleerde forecast vs oracle forecast

Vraag: hoeveel van DQN's prestaties hangt af van forecast-kwaliteit? Met oracle-forecast (ground-truth demand uit features.parquet) zou DQN's `forecast_top` macro de juiste hot zones aanwijzen ipv het Transformer-undershoot pattern.

In [7]:
ablation_rows = []
dqn_factory = AGENT_FACTORIES['dqn']
for d in DATES:
    oracle = forecaster.oracle_forecast_day(d)
    for seed in range(N_SEEDS):
        # DQN with predicted forecast
        res_pred = run_episode(dqn_factory, env, d, seed, 'dqn_pred', override_forecast=None)
        ablation_rows.append(res_pred)
        # DQN with oracle forecast
        res_oracle = run_episode(dqn_factory, env, d, seed, 'dqn_oracle', override_forecast=oracle)
        ablation_rows.append(res_oracle)
    print(f'  ablation done for {d}')
ablation_df = pd.DataFrame(ablation_rows)
abl_pivot = ablation_df.pivot_table(index='agent', columns='date', values='answered_calls', aggfunc='mean').round(1)
abl_pivot['mean'] = abl_pivot.mean(axis=1).round(1)
print('\nAblation: answered_calls (mean over 10 seeds)')
abl_pivot

  ablation done for 2026-04-30


  ablation done for 2026-05-01


  ablation done for 2026-05-02

Ablation: answered_calls (mean over 10 seeds)


date,2026-04-30,2026-05-01,2026-05-02,mean
agent,,,,
dqn_oracle,415.3,881.3,501.0,599.2
dqn_pred,137.0,322.7,147.7,202.5


In [8]:
# Per-day lift from oracle
lifts = []
for d in DATES:
    p = ablation_df[(ablation_df.agent=='dqn_pred')  & (ablation_df.date==d.isoformat())]['answered_calls'].mean()
    o = ablation_df[(ablation_df.agent=='dqn_oracle')& (ablation_df.date==d.isoformat())]['answered_calls'].mean()
    lifts.append({'date': d.isoformat(), 'pred': p, 'oracle': o, 'lift_abs': o-p, 'lift_pct': 100*(o-p)/p if p else 0})
lift_df = pd.DataFrame(lifts).round(2)
lift_df

,date,pred,oracle,lift_abs,lift_pct
0,2026-04-30,137.00,415.33,278.33,203.16
1,2026-05-01,322.67,881.33,558.67,173.14
2,2026-05-02,147.67,501.00,353.33,239.28


## Visualisatie

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# (a) Bar chart of mean answered calls per (agent, day)
ax = axes[0]
agents_order = ['random', 'greedy', 'historical', 'q_learning', 'dqn']
x = np.arange(len(agents_order))
width = 0.25
for i, d in enumerate(DATES):
    means = [main_df[(main_df.agent==a) & (main_df.date==d.isoformat())]['answered_calls'].mean() for a in agents_order]
    ax.bar(x + (i-1)*width, means, width, label=d.isoformat())
ax.set_xticks(x, agents_order, rotation=20)
ax.set_ylabel('Mean answered calls')
ax.set_title('Agent vergelijking — gemiddelde sales (10 seeds)')
ax.legend(title='date')
ax.grid(True, axis='y', alpha=0.3)

# (b) Ablation: DQN pred vs oracle
ax = axes[1]
x2 = np.arange(len(DATES))
p_means = [ablation_df[(ablation_df.agent=='dqn_pred')   & (ablation_df.date==d.isoformat())]['answered_calls'].mean() for d in DATES]
o_means = [ablation_df[(ablation_df.agent=='dqn_oracle') & (ablation_df.date==d.isoformat())]['answered_calls'].mean() for d in DATES]
ax.bar(x2 - 0.2, p_means, 0.4, label='DQN + predicted forecast', color='#1f77b4')
ax.bar(x2 + 0.2, o_means, 0.4, label='DQN + oracle forecast', color='#d62728')
ax.set_xticks(x2, [d.isoformat() for d in DATES])
ax.set_ylabel('Mean answered calls')
ax.set_title('Ablation: forecast quality lift')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(FIGURES / 'agent_comparison.png', dpi=110, bbox_inches='tight')
plt.show()

C:\Users\ralph\AppData\Local\Temp\ipykernel_38508\1459495465.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Key insight

**1. Forecast-kwaliteit is veruit de grootste bottleneck.**

| | Mean answered_calls (3 dagen × 3 seeds) |
|---|---:|
| DQN met geleerde Transformer-forecast | 202.5 |
| **DQN met oracle (ground-truth) forecast** | **599.2** |
| **Lift** | **+196%** (≈ 3× zoveel sales) |

Per dag: +203% (30/4), +173% (1/5), +239% (2/5). De Transformer-magnitude-onderschatting uit [02_forecast.ipynb](02_forecast.ipynb) limiteert de agent-prestaties harder dan welke architectuur-keuze ook in agents zelf. **Een betere forecaster zou meer waarde leveren dan een betere agent.**

**2. Tabular Q ≥ DQN op deze MDP — function approximation is overkill bij 4 macros.**

| Agent | Mean answered_calls |
|---:|---:|
| q_learning | **200.8** |
| dqn | 187.1 |
| historical | 118.0 |
| greedy | 115.2 |
| random | 76.3 |

Tabular Q wint met +7% over DQN, ondanks (of door?) z'n simpelere structuur. Lessen: bij 4 hand-gepickte macro-acties is een 12-state Q-table genoeg om de optimale policy te capturen, en DQN's continue state-rep voegt niets toe maar introduceert wel meer variantie. Het eerdere single-seed-resultaat (DQN > Q) zat binnen seed-noise — multi-seed gemiddelde geeft een eerlijker beeld.

**3. Trade-off response-time vs total-sales: greedy chase'd calls, RL agents poolen demand.**

| Agent | Mean response (min) | Mean answered_calls |
|---|---:|---:|
| greedy | **23** | 115 |
| q_learning | 52 | 201 |
| dqn | 75 | 187 |
| historical | 121 | 118 |
| random | 147 | 76 |

Greedy is sneller bij individuele calls maar haalt minder totale sales. Q/DQN nemen meer tijd per call maar staan in betere zones, wat globaal meer demand realiseert. Voor een ijswagen-business is total-sales waarschijnlijk belangrijker dan per-call-response.

**Conclusie voor de fiche**

De grootste hefboom voor verbeterde agent-prestaties zit in de **forecast-kwaliteit**, niet in de agent-architectuur. Met de huidige Transformer (~50% magnitude-undershoot, zie [02_forecast.ipynb](02_forecast.ipynb)) lopen alle agents tegen dezelfde wand op. Een forecaster-fix (squared-error retraining, log-target, of meer dagen data — zie [docs/limitations.md](../docs/limitations.md)) zou ~3× meer sales realiseren dan welke RL-tweak ook.

---

**Volgende**: [05_results.ipynb](05_results.ipynb) — de hero figuren voor fiche/defense, gebouwd op de cijfers uit dit notebook + [results/eval_summary.csv](../results/eval_summary.csv).